# XGBoost Baseline - binary classification: fire vs megafire

Input dataset: `data/processed/conaf_enriched_latest.parquet`  
Target variable: `is_megafire`, where `1` means `superficie_quemada_total_ha >= MEGAFIRE_HA_THRESHOLD` and `0` otherwise.

Training uses every non-burned-area column after converting datetime and categorical columns into model-usable numeric features.


In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PROCESSED = Path("../data/processed")
DATA_MODELS = Path("../data/models")
DATA_MODELS.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET_COL = "superficie_quemada_total_ha"
LABEL_COL = "is_megafire"
MEGAFIRE_HA_THRESHOLD = 1200

XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}


## 1. Load dataset

In [ ]:
df = pd.read_parquet(DATA_PROCESSED / "conaf_enriched_latest.parquet")
print(f"{len(df):,} rows x {df.shape[1]} columns")
df.dtypes.value_counts()


## 2. Target variable exploration

Use this cell to decide the megafire threshold before filling in `MEGAFIRE_HA_THRESHOLD`.

In [ ]:
s = pd.to_numeric(df[TARGET_COL], errors="coerce").dropna()

print(s.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
s.clip(upper=s.quantile(0.99)).hist(bins=50, ax=axes[0])
axes[0].set_title("Distribution without extreme outliers")
axes[0].set_xlabel("ha")
np.log1p(s).hist(bins=50, ax=axes[1])
axes[1].set_title("log1p distribution")
axes[1].set_xlabel("log(ha + 1)")
plt.tight_layout()
plt.show()


## 3. Target variable definition

In [ ]:
assert MEGAFIRE_HA_THRESHOLD is not None, "Set MEGAFIRE_HA_THRESHOLD in the configuration cell"

df = df.dropna(subset=[TARGET_COL]).copy()
df[LABEL_COL] = (pd.to_numeric(df[TARGET_COL], errors="coerce") >= MEGAFIRE_HA_THRESHOLD).astype(int)

burnt_area_cols = [
    col for col in df.columns
    if col.startswith("superficie_quemada_") or "burnt_area" in col.lower() or "burned_area" in col.lower()
]
df = df.drop(columns=burnt_area_cols)

balance = df[LABEL_COL].value_counts().reindex([0, 1], fill_value=0)
scale_pos_weight = balance.loc[0] / balance.loc[1] if balance.loc[1] else 1.0

print(balance.rename(index={0: "fire", 1: "megafire"}))
print(f"\nRatio negatives/positives: {scale_pos_weight:.2f}")
print(f"Removed burned-area columns: {burnt_area_cols}")


## 4. Feature selection

In [ ]:
# Use every remaining dataframe column except the target label.
# Burned-area columns were already removed in the previous cell.
model_df = df.drop(columns=[LABEL_COL]).copy()

datetime_cols = [col for col in model_df.columns if pd.api.types.is_datetime64_any_dtype(model_df[col])]
for col in datetime_cols:
    ts = pd.to_datetime(model_df[col], errors="coerce")
    model_df[f"{col}_year"] = ts.dt.year
    model_df[f"{col}_month"] = ts.dt.month
    model_df[f"{col}_day"] = ts.dt.day
    model_df[f"{col}_hour"] = ts.dt.hour
    model_df[f"{col}_day_of_year"] = ts.dt.day_of_year
    model_df = model_df.drop(columns=[col])

categorical_cols = [
    col for col in model_df.columns
    if not pd.api.types.is_numeric_dtype(model_df[col])
]
for col in categorical_cols:
    model_df[col] = pd.Categorical(model_df[col]).codes

bool_cols = [col for col in model_df.columns if pd.api.types.is_bool_dtype(model_df[col])]
for col in bool_cols:
    model_df[col] = model_df[col].astype(int)

feature_cols = list(model_df.columns)
X = model_df
y = df[LABEL_COL]

burnt_area_leaks = [
    col for col in feature_cols
    if col.startswith("superficie_quemada_") or "burnt_area" in col.lower() or "burned_area" in col.lower()
]
assert not burnt_area_leaks, f"Burned-area leakage in features: {burnt_area_leaks}"

print(f"Features: {len(feature_cols)}")
print(feature_cols)


## 5. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"Positives in train: {y_train.sum()} ({y_train.mean():.1%})")
print(f"Positives in test:  {y_test.sum()} ({y_test.mean():.1%})")


## 6. XGBoost training

In [ ]:
model = xgb.XGBClassifier(
    **XGB_PARAMS,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    random_state=RANDOM_STATE,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)


## 7. Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC: {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["fire", "megafire"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["fire", "megafire"],
    ax=axes[0],
)
axes[0].set_title("Confusion matrix")

# Feature importance (top 20)
importances = pd.Series(model.feature_importances_, index=feature_cols).nlargest(20)
importances.sort_values().plot.barh(ax=axes[1])
axes[1].set_title("Feature importance (top 20)")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

## 8. Save model and metadata

In [ ]:
model_path = DATA_MODELS / "xgboost_baseline.json"
meta_path = DATA_MODELS / "xgboost_baseline_meta.json"

model.save_model(model_path)

meta = {
    "threshold_ha": MEGAFIRE_HA_THRESHOLD,
    "label_col": LABEL_COL,
    "removed_burned_area_features": burnt_area_cols,
    "features": list(X.columns),
    "n_features": len(feature_cols),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "auc_roc": float(auc),
    "xgb_params": {**XGB_PARAMS, "scale_pos_weight": scale_pos_weight},
}
meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False))

print(f"Model saved to: {model_path}")
print(f"Metadata saved to: {meta_path}")
